# Cleaning Data

## Importing Libraries

In [1]:
# Importing necessary libraries
import pandas as pd
import numpy as np

## Reading the Original Data

In [ ]:
# Reading in both files. There probably is a better way of reading them in, but I'm tight on time so that will have to do
marchApril = pd.read_csv(
    "C:/Users/sKALa/Repos/Semester 2/Data Visualisation and Insight/CAs/CA 2/Data-Visualisation-and-Insight---CA-2/data/original/data0304.csv")
aprilMay = pd.read_csv(
    "C:/Users/sKALa/Repos/Semester 2/Data Visualisation and Insight/CAs/CA 2/Data-Visualisation-and-Insight---CA-2/data/original/data0405.csv")

## Combining Datasets

In [5]:
# Combining both datasets to store them as one df
df = pd.concat([marchApril, aprilMay], ignore_index = True)

## Cleaning the Records

In [7]:
# Converting the ActivityDate variable to the datetime type
df["ActivityDate"] = pd.to_datetime(df["ActivityDate"])

In [8]:
# Sorting the data by user ID and date
df = df.sort_values(["Id", "ActivityDate"])

In [9]:
# Removing duplicate columns just in case some records overlapped in April
df = df.drop_duplicates(subset = ["Id", "ActivityDate"], keep = "first")

## Adding new Columns

In [ ]:
# Weekday and month columns
df["Weekday"] = df["ActivityDate"].dt.day_name()
df["Month"] = df["ActivityDate"].dt.month_name()

In [ ]:
# Extra activity columns
df["ActiveMinutes"] = (df["VeryActiveMinutes"] + df["FairlyActiveMinutes"] + df["LightlyActiveMinutes"])
df["TotalRecordedMinutes"] = df["ActiveMinutes"] + df["SedentaryMinutes"]
df["ActiveHours"] = df["ActiveMinutes"] / 60
df["SedentaryHours"] = df["SedentaryMinutes"] / 60

In [12]:
# Percentage of recorded time spent active
df["ActivePercentage"] = np.where(df["TotalRecordedMinutes"] > 0, (df["ActiveMinutes"] / df["TotalRecordedMinutes"]) * 100, np.nan)

In [13]:
# Calories per step
df["CaloriesPerStep"] = np.where(df["TotalSteps"] > 0, df["Calories"] / df["TotalSteps"], np.nan)

In [ ]:
# Calories per kilometre/mile unit depending on original Fitbit distance
df["CaloriesPerDistance"] = np.where(df["TotalDistance"] > 0, df["Calories"] / df["TotalDistance"], np.nan)

In [14]:
# Step category
df["StepCategory"] = pd.cut(df["TotalSteps"], bins = [-1, 4999, 7499, 9999, float("inf")], labels = ["Low activity", "Light activity",
                           "Moderately active", "Highly active"])

In [15]:
# Weekend/weekday column
df["DayType"] = np.where(df["Weekday"].isin(["Saturday", "Sunday"]), "Weekend", "Weekday")

## Saving the Dataset

In [ ]:
# Resetting index after cleaning
df = df.reset_index(drop = True)

# Saving cleaned dataset
df.to_csv(
    "C:/Users/sKALa/Repos/Semester 2/Data Visualisation and Insight/CAs/CA 2/Data-Visualisation-and-Insight---CA-2/data/preprocessed/cleanData.csv",
    index = False)

## Checking Results

In [ ]:
# Check result
print("Cleaned dataset saved as cleaned_fitbit_data.csv")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("Unique users:", df["Id"].nunique())
print("Date range:", df["ActivityDate"].min(), "to", df["ActivityDate"].max())

df.head()